In [6]:
# churn_prediction_project.py - COMPLETE CHURN PROJECT
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif

# 1. LOAD YOUR DATASET
print("=== CUSTOMER CHURN PREDICTION PROJECT ===")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print("\nTarget distribution:")
print(train_df['Customer Status'].value_counts(normalize=True))

# 2. PREPROCESSING
print("\nPreprocessing...")
numeric_features = ['Age', 'Avg Monthly GB Download', 'Monthly Charge', 
                   'Tenure in Months', 'Churn Score', 'Satisfaction Score']
categorical_features = ['Contract', 'Payment Method', 'Internet Service', 
                       'Gender', 'Offer', 'Paperless Billing']

# Clean data
X_train_full = train_df[numeric_features + categorical_features].fillna(0)
y_train_full = train_df['Customer Status']
X_test = test_df[numeric_features + categorical_features].fillna(0)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])

# 3. TRAIN MODELS
print("\nTraining models...")
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=200)
}

trained_models = {}
results = {}

for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    
    results[name] = accuracy
    trained_models[name] = pipe
    print(f"{name}: {accuracy:.3f}")

# Ensemble
print("\nTraining ensemble...")
ensemble = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])
ensemble.fit(X_train, y_train)
ensemble_acc = accuracy_score(y_val, ensemble.predict(X_val))
print(f"Ensemble: {ensemble_acc:.3f}")

# 4. VISUALIZATIONS
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Model Comparison
model_names = list(results.keys()) + ['Ensemble']
model_scores = list(results.values()) + [ensemble_acc]
sns.barplot(x=model_scores, y=model_names, ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Model Comparison (Validation Accuracy)')

# 2. Confusion Matrix (Ensemble)
cm = confusion_matrix(y_val, ensemble.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1])
axes[0,1].set_title('Confusion Matrix (Ensemble)')

# 3. Feature Importance
rf_model = trained_models['Random Forest']
feature_names = (numeric_features + 
                list(rf_model.named_steps['preprocess']
                     .named_transformers_['cat'].get_feature_names_out(categorical_features)))
importances = rf_model.named_steps['model'].feature_importances_
top10_idx = np.argsort(importances)[-10:]
sns.barplot(x=importances[top10_idx], y=[feature_names[i] for i in top10_idx], ax=axes[0,2])
axes[0,2].set_title('Top 10 Feature Importances')

# 4. Churn by Contract
sns.countplot(data=train_df, x='Contract', hue='Customer Status', ax=axes[1,0])
axes[1,0].set_title('Churn by Contract Type')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Monthly Charge Distribution
sns.boxplot(data=train_df, x='Customer Status', y='Monthly Charge', ax=axes[1,1])
axes[1,1].set_title('Monthly Charge by Status')

# 6. Tenure Distribution
sns.histplot(data=train_df, x='Tenure in Months', hue='Customer Status', 
             bins=20, ax=axes[1,2], multiple='stack')
axes[1,2].set_title('Tenure Distribution by Status')

plt.tight_layout()
plt.savefig('churn_visualizations.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. TEST PREDICTIONS
print("\nGenerating test predictions...")
test_preds = ensemble.predict(X_test)

submission = pd.DataFrame({
    'Customer ID': test_df['Customer ID'],
    'prediction_label': test_preds
})
submission.to_csv('churn_predictions.csv', index=False)

print("Predictions saved: churn_predictions.csv")
print("\nTest prediction distribution:")
print(pd.Series(test_preds).value_counts())

# 6. FINAL REPORT
print("\n=== FINAL RESULTS ===")
print(f"Best Model: Ensemble (Accuracy: {ensemble_acc:.3f})")
print("\nClassification Report (Validation):")
print(classification_report(y_val, ensemble.predict(X_val)))

print("\nPROJECT COMPLETE!")
print("Saved files:")
print("- churn_visualizations.png")
print("- churn_predictions.csv")


=== CUSTOMER CHURN PREDICTION PROJECT ===
Train: (4225, 52), Test: (1409, 52)

Target distribution:
Customer Status
Stayed     0.670296
Churned    0.265325
Joined     0.064379
Name: proportion, dtype: float64

Preprocessing...

Training models...


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']